In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
import sys

# Add src to path
sys.path.insert(0, '../src')
from preprocessing import DRPreprocessor

print("✅ Libraries imported successfully!")

## 1. Dataset Overview

In [ ]:
# Load training labels
aptos_csv = Path('../data/aptos2019/train.csv')
if aptos_csv.exists():
    train_df = pd.read_csv(aptos_csv)
    IMAGE_DIR = Path('../data/aptos2019/train_images')
else:
    raise FileNotFoundError(
        "APTOS dataset not found. Run: python scripts/download_dataset.py"
    )

print("Dataset Shape:", train_df.shape)
print("\nSample Data:")
display(train_df.head(10))

print("\nClass Distribution:")
class_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
for i, name in enumerate(class_names):
    count = len(train_df[train_df['diagnosis'] == i])
    pct = 100 * count / len(train_df)
    print(f"  Class {i} ({name}): {count} images ({pct:.1f}%)")

In [ ]:
# Visualize class distribution
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#8e44ad']
counts = train_df['diagnosis'].value_counts().sort_index().values

bars = ax.bar(class_names, counts, color=colors, edgecolor='black', linewidth=1.5)

ax.set_xlabel('DR Severity Class', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Images', fontsize=12, fontweight='bold')
ax.set_title('Diabetic Retinopathy - Class Distribution', fontsize=14, fontweight='bold')

# Add value labels
for bar, val in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
            str(val), ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 2. Sample Images - All DR Classes

In [ ]:
# Display sample images from each class
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle('Sample Retinal Images - All DR Severity Classes', fontsize=14, fontweight='bold')

for i, (ax, class_name) in enumerate(zip(axes, class_names)):
    sample = train_df[train_df['diagnosis'] == i].iloc[0]
    img_path = IMAGE_DIR / f"{sample['id_code']}.png"
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    ax.imshow(img)
    ax.set_title(f'Class {i}\n{class_name}', fontsize=11, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3. Preprocessing Pipeline Demonstration

In [ ]:
# Initialize the preprocessor
preprocessor = DRPreprocessor(
    img_size=512,
    ben_graham_sigma=10,
    clahe_clip_limit=2.0,
    clahe_tile_size=8
)

print("🔧 DRPreprocessor Configuration:")
print(f"   Image Size: {preprocessor.img_size}x{preprocessor.img_size}")
print(f"   Ben Graham Sigma: {preprocessor.ben_graham_sigma}")
print(f"   CLAHE Clip Limit: {preprocessor.clahe_clip_limit}")
print(f"   CLAHE Tile Size: {preprocessor.clahe_tile_size}")

In [ ]:
# Demonstrate preprocessing steps on a single image
sample_row = train_df[train_df['diagnosis'] == 2].iloc[0]
sample_img_path = IMAGE_DIR / f"{sample_row['id_code']}.png"
print(f"Processing: {sample_img_path.name}")

# Load original image
img = cv2.imread(str(sample_img_path))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Apply each step
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Preprocessing Pipeline Steps', fontsize=14, fontweight='bold')

# Step 1: Original
axes[0, 0].imshow(img)
axes[0, 0].set_title('1. Original Image', fontsize=11)
axes[0, 0].axis('off')

# Step 2: Crop black borders
img_cropped = preprocessor.crop_black_borders(img)
axes[0, 1].imshow(img_cropped)
axes[0, 1].set_title('2. After Border Cropping', fontsize=11)
axes[0, 1].axis('off')

# Step 3: Resize
img_resized = preprocessor.resize(img_cropped)
axes[0, 2].imshow(img_resized)
axes[0, 2].set_title(f'3. Resized to {preprocessor.img_size}x{preprocessor.img_size}', fontsize=11)
axes[0, 2].axis('off')

# Step 4: Ben Graham preprocessing
img_ben = preprocessor.ben_graham_preprocessing(img_resized)
axes[1, 0].imshow(img_ben)
axes[1, 0].set_title('4. Ben Graham Preprocessing', fontsize=11)
axes[1, 0].axis('off')

# Step 5: Circle crop
img_circle = preprocessor.circle_crop(img_ben)
axes[1, 1].imshow(img_circle)
axes[1, 1].set_title('5. Circle Cropped', fontsize=11)
axes[1, 1].axis('off')

# Step 6: CLAHE
img_clahe = preprocessor.apply_clahe(img_circle)
axes[1, 2].imshow(img_clahe)
axes[1, 2].set_title('6. After CLAHE Enhancement', fontsize=11)
axes[1, 2].axis('off')

plt.tight_layout()
plt.savefig('../results/figures/preprocessing_steps_demo.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Saved: results/figures/preprocessing_steps_demo.png")

## 4. Before vs After Preprocessing Comparison

In [ ]:
# Compare raw vs processed for all classes
fig, axes = plt.subplots(5, 2, figsize=(12, 25))
fig.suptitle('Raw vs Preprocessed Images - All Classes', fontsize=14, fontweight='bold', y=1.01)

for i, class_name in enumerate(class_names):
    sample = train_df[train_df['diagnosis'] == i].iloc[0]
    raw_path = IMAGE_DIR / f"{sample['id_code']}.png"
    raw_img = cv2.imread(str(raw_path))
    raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)

    proc_img = preprocessor.preprocess(raw_img.copy(), normalize=False).astype(np.uint8)
    
    axes[i, 0].imshow(raw_img)
    axes[i, 0].set_title(f'Class {i}: {class_name} (Raw)', fontsize=11)
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(proc_img)
    axes[i, 1].set_title(f'Class {i}: {class_name} (Preprocessed)', fontsize=11)
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

## 5. Histogram Analysis

In [ ]:
# Compare histograms before and after preprocessing
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Histogram Analysis - Effect of Preprocessing', fontsize=14, fontweight='bold')

sample_row = train_df[train_df['diagnosis'] == 2].iloc[0]
sample_path = IMAGE_DIR / f"{sample_row['id_code']}.png"

# Raw image
raw_img = cv2.imread(str(sample_path))
raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)

# Processed image
proc_img = preprocessor.preprocess(raw_img, normalize=False)

# Raw image display
axes[0, 0].imshow(raw_img)
axes[0, 0].set_title('Raw Image', fontsize=11)
axes[0, 0].axis('off')

# Raw histogram per channel
for c, color in enumerate(['red', 'green', 'blue']):
    axes[0, 1].hist(raw_img[:,:,c].flatten(), bins=50, color=color, alpha=0.5, label=color.capitalize())
axes[0, 1].set_title('Raw - RGB Histogram', fontsize=11)
axes[0, 1].legend()
axes[0, 1].set_xlabel('Pixel Value')
axes[0, 1].set_ylabel('Frequency')

# Raw green channel (most important for DR)
axes[0, 2].hist(raw_img[:,:,1].flatten(), bins=50, color='green', alpha=0.7)
axes[0, 2].set_title('Raw - Green Channel', fontsize=11)
axes[0, 2].set_xlabel('Pixel Value')

# Processed image display
axes[1, 0].imshow(proc_img.astype(np.uint8))
axes[1, 0].set_title('Preprocessed Image', fontsize=11)
axes[1, 0].axis('off')

# Processed histogram per channel
for c, color in enumerate(['red', 'green', 'blue']):
    axes[1, 1].hist(proc_img[:,:,c].flatten(), bins=50, color=color, alpha=0.5, label=color.capitalize())
axes[1, 1].set_title('Preprocessed - RGB Histogram', fontsize=11)
axes[1, 1].legend()
axes[1, 1].set_xlabel('Pixel Value')
axes[1, 1].set_ylabel('Frequency')

# Processed green channel
axes[1, 2].hist(proc_img[:,:,1].flatten(), bins=50, color='blue', alpha=0.7)
axes[1, 2].set_title('Preprocessed - Green Channel', fontsize=11)
axes[1, 2].set_xlabel('Pixel Value')

plt.tight_layout()
plt.savefig('../results/figures/histogram_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Saved: results/figures/histogram_analysis.png")

## 6. Summary Statistics

In [ ]:
# Print summary of processed images
print("=" * 50)
print("📊 PREPROCESSING SUMMARY")
print("=" * 50)

all_images = list(IMAGE_DIR.glob('*.png'))
print(f"\nTotal Training Images: {len(all_images)}")
print(f"Labels in CSV: {len(train_df)}")

sample_raw = cv2.imread(str(all_images[0]))
sample_proc = preprocessor.preprocess(cv2.cvtColor(sample_raw, cv2.COLOR_BGR2RGB), normalize=False)

print(f"\nRaw Image Size: {sample_raw.shape}")
print(f"Processed Image Size: {sample_proc.shape}")
print(f"Target Size: {preprocessor.img_size}x{preprocessor.img_size}")

print("\nPreprocessing pipeline ready!")
print("\nPipeline steps:")
print("   1. Black border cropping")
print("   2. Resize to 512x512")
print("   3. Ben Graham contrast enhancement")
print("   4. Circular mask")
print("   5. CLAHE adaptive histogram equalization")
print("   6. ImageNet normalization")
print("   4. Build and train the model")

---

## 📝 Notes

### Preprocessing Techniques Used:

1. **Black Border Cropping**: Removes the black regions around the retinal image

2. **Ben Graham's Preprocessing**: Enhances local contrast by subtracting a Gaussian-blurred version of the image. This:
   - Normalizes lighting variations
   - Enhances blood vessels visibility
   - Makes lesions more prominent

3. **Circle Cropping**: Applies a circular mask to focus on the retinal region

4. **CLAHE (Contrast Limited Adaptive Histogram Equalization)**: Further enhances local contrast while limiting noise amplification

### References:
- Ben Graham (Kaggle DR Competition Winner, 2015)
- Gulshan et al., JAMA 2016
- Wang et al., IEEE Access 2024